Simulamos nulos y hacemos interpolación lineal

In [8]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado. Asegúrate de que esté en la carpeta 'CSVs' dentro de tu proyecto.")
    df_air_quality = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv en la ruta esperada.")
    print("Creando DataFrame de ejemplo para continuar con la ejecución del script...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([
                _date, _city_ex,
                200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5,
                50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20,
                40+np.random.rand()*20
            ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
if 'CO2' in df_air_quality.columns:
    df_air_quality = df_air_quality.drop(columns=['CO2'])
    print("Columna 'CO2' descartada.")

# --- Parámetros de Simulación e Imputación ---
CONTAMINANT_AND_AQI_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

# --- Rutas de Salida (dentro de AIR_QUALITY) ---
PROJECT_ROOT_DIR = os.getcwd()

SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'

# AIR_QUALITY/imputaciones/interpolacionLineal/metricas/
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
# AIR_QUALITY/imputaciones/interpolacionLineal/CSVs/
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)
os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True) # Descomentado

# --- Funciones Auxiliares (sin cambios en la lógica interna) ---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = []
    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']
    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0 :
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length = config['length']
        num_blocks_ratio = config.get('num_blocks_ratio', 0.01)
        num_blocks = max(1, int( (n_rows * num_blocks_ratio) / block_length if block_length > 0 else n_rows * num_blocks_ratio ))
        for _ in range(num_blocks):
            if n_rows < block_length: break
            if (n_rows - block_length) < 0: continue
            start_pos = random.randint(0, n_rows - block_length)
            current_block_positions = list(range(start_pos, start_pos + block_length))
            row_indices_to_nan_positions.extend(current_block_positions)
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(CONTAMINANT_AND_AQI_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan:
             df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    df_original_city_reset = df_original_city_input.copy().reset_index(drop=True)
    cols_to_check_mask = [col for col in CONTAMINANT_AND_AQI_COLS if col in df_with_nans.columns and col in df_original_city_reset.columns]
    if not cols_to_check_mask:
        introduced_nan_mask_df = pd.DataFrame(index=df_with_nans.index)
    else:
        introduced_nan_mask_df = df_with_nans[cols_to_check_mask].isnull() & ~df_original_city_reset[cols_to_check_mask].isnull()
    return df_with_nans, introduced_nan_mask_df

def calculate_metrics(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    mae_vals, rmse_vals = {}, {}
    for col in CONTAMINANT_AND_AQI_COLS:
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan
            continue
        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values
        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan
            continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan
            continue
        try:
            mae_vals[col] = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse_vals[col] = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
        except Exception as e:
            print(f"Error calculando métricas para {col}: {e}")
            mae_vals[col], rmse_vals[col] = np.nan, np.nan
    return mae_vals, rmse_vals

# --- Script Principal ---
all_detailed_metrics_results = []
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Carga tus datos primero.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")

    # Directorios para datos simulados de esta ciudad
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)

    # Directorios para resultados de imputación de esta ciudad
    city_metrics_file_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_{INTERPOLATION_METHOD_NAME}_{city_name_safe}.csv')
    city_imputed_csv_dir_root = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe) # Para CSVs imputados
    os.makedirs(city_imputed_csv_dir_root, exist_ok=True)


    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty:
        print(f"  No hay datos para la ciudad {city_name_original}. Omitiendo.")
        continue
    if df_city_original_for_sim[CONTAMINANT_AND_AQI_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill para ground truth de simulación.")
        for col_fill in CONTAMINANT_AND_AQI_COLS:
             if col_fill in df_city_original_for_sim.columns:
                df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[CONTAMINANT_AND_AQI_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs preexistentes en {city_name_original} tras ffill/bfill. Omitiendo.")
            continue
    df_city_original_ground_truth_reset = df_city_original_for_sim.copy().reset_index(drop=True)
    city_scenario_summary_metrics = []

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Escenario de Simulación de NaNs: {scenario_name_safe}")
        
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)

        scenario_city_imputed_csv_dir_specific = os.path.join(city_imputed_csv_dir_root, scenario_name_safe) # Para CSVs imputados
        os.makedirs(scenario_city_imputed_csv_dir_specific, exist_ok=True)


        scenario_rep_maes = {var: [] for var in CONTAMINANT_AND_AQI_COLS}
        scenario_rep_rmses = {var: [] for var in CONTAMINANT_AND_AQI_COLS}

        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time_ns() + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans_reset, introduced_nan_mask_df_reset = introduce_nans_by_row(
                df_city_original_for_sim, scenario, random_seed_input=current_seed
            )
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans_reset.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
            
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in CONTAMINANT_AND_AQI_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None
                    )
            for col in CONTAMINANT_AND_AQI_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()
            
            # Guardar el CSV imputado
            imputed_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_imputed_linear.csv"
            df_city_imputed_linear_reset.to_csv(os.path.join(scenario_city_imputed_csv_dir_specific, imputed_filename), index=False)

            mae_scores, rmse_scores = calculate_metrics(
                df_city_original_ground_truth_reset,
                df_city_imputed_linear_reset,
                introduced_nan_mask_df_reset
            )
            for var_name in CONTAMINANT_AND_AQI_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 'MAE': mae_scores.get(var_name, np.nan), 'RMSE': rmse_scores.get(var_name, np.nan)
                })
                if not np.isnan(mae_scores.get(var_name, np.nan)):
                    scenario_rep_maes[var_name].append(mae_scores.get(var_name))
                if not np.isnan(rmse_scores.get(var_name, np.nan)):
                    scenario_rep_rmses[var_name].append(rmse_scores.get(var_name))
        
        avg_mae_aqi_scenario = np.nanmean(scenario_rep_maes.get('AQI', [np.nan]))
        avg_rmse_aqi_scenario = np.nanmean(scenario_rep_rmses.get('AQI', [np.nan]))
        print(f"    Promedio para Escenario '{scenario_name_safe}' (AQI): MAE={avg_mae_aqi_scenario:.4f}, RMSE={avg_rmse_aqi_scenario:.4f}")
        
        for var_name in CONTAMINANT_AND_AQI_COLS:
            city_scenario_summary_metrics.append({
                'Scenario': scenario_name_safe, 'Variable': var_name,
                'Avg_MAE': np.nanmean(scenario_rep_maes.get(var_name, [np.nan])),
                'Avg_RMSE': np.nanmean(scenario_rep_rmses.get(var_name, [np.nan])),
                'Std_MAE': np.nanstd(scenario_rep_maes.get(var_name, [np.nan])),
                'Std_RMSE': np.nanstd(scenario_rep_rmses.get(var_name, [np.nan]))
            })
    if city_scenario_summary_metrics:
        df_city_metrics_summary = pd.DataFrame(city_scenario_summary_metrics)
        df_city_metrics_summary.to_csv(city_metrics_file_path, index=False)
        print(f"  Resumen de métricas promedio para {city_name_original} guardado en: {city_metrics_file_path}")

if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    global_detailed_metrics_file = os.path.join(METRICS_OUTPUT_DIR, f'ALL_metrics_detailed_{INTERPOLATION_METHOD_NAME}.csv')
    df_all_detailed_metrics.to_csv(global_detailed_metrics_file, index=False)
    print(f"\nTodas las métricas detalladas guardadas en: {global_detailed_metrics_file}")
    print("\nContenido del DataFrame de Métricas Detalladas (primeras filas):")
    print(df_all_detailed_metrics.head())

end_time_script = time.time()
print(f"\nProceso completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Resúmenes de métricas promedio por ciudad guardados en: {METRICS_OUTPUT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Escenario de Simulación de NaNs: RandomRows_10percent
    Promedio para Escenario 'RandomRows_10percent' (AQI): MAE=0.6762, RMSE=1.2196
  Escenario de Simulación de NaNs: RandomRows_20percent
    Promedio para Escenario 'RandomRows_20percent' (AQI): MAE=0.7997, RMSE=1.4205
  Escenario de Simulación de NaNs: BlockRows_2h
    Promedio para Escenario 'BlockRows_2h' (AQI): MAE=0.9912, RMSE=1.6295
  Escenario de Simulación de NaNs: BlockRows_4h
    Promedio para Escenario 'BlockRows_4h' (AQI): MAE=1.8926, RMSE=2.7344
  Escenario de Simulación de NaNs: BlockRows_8h
    Promedio para Escenario 'BlockRows_8h' (AQI): MAE=3.8853, RMSE=5.2101
  Escenario de Simulación de NaNs: BlockRows_12h
    Promedio para Escenario 'BlockRows_12h' (AQI): MAE=5.0423, RMSE=6.7134
  Escenario de Simulación de NaNs: BlockRows_24h
    Promedio para Escenario 'BlockRows_24h' (AQI): MAE=9.